# Wavelet packet denoising with the `fcwt2` Python API

This notebook mirrors [`examples/wavelet_packet_denoiser.rs`](../examples/wavelet_packet_denoiser.rs), but keeps the workflow in Python so the intermediate steps are easy to inspect visually.

The pipeline is:

1. Build a clean synthetic signal.
2. Add deterministic noise.
3. Select a wavelet basis from coarsest-detail PDF shape.
4. Decompose with a wavelet packet transform.
5. Soft-threshold detail-bearing packet leaves.
6. Reconstruct through `fcwt2.WaveletPacketTree.from_leaves(...)`.

## Setup

If these APIs are not in your installed PyPI package yet, install the local checkout with `maturin develop` from the repository root.

In [ ]:
# Released package path:
# %pip install -q fcwt2 numpy matplotlib

# Local checkout path, run from the repository root or adjust the manifest path:
# %pip install -q maturin numpy matplotlib
# !maturin develop --manifest-path ../Cargo.toml

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import fcwt2

plt.rcParams.update(
    {
        "figure.figsize": (12, 4),
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

n = 256
levels = 4
noise_amplitude = 0.20
candidates = ["haar", "db2", "db4", "db8", "sym4", "sym8"]

## Match the Rust example signal

The signal combines a smooth tone, a localized oscillatory burst, and a step. The noise generator is deterministic so plots and RMSE values are reproducible.

In [ ]:
def synthetic_signal(n):
    sample = np.arange(n, dtype=np.float32)
    t = sample / np.float32(n)
    slow = np.sin(2.0 * np.pi * 5.0 * t)
    burst_window = np.exp(-0.5 * ((t - 0.58) / 0.05) ** 2)
    burst = burst_window * np.sin(2.0 * np.pi * 42.0 * t)
    step = np.where(t > 0.35, 0.35, 0.0)
    return (0.65 * slow + 0.8 * burst + step).astype(np.float32)


def add_deterministic_noise(signal, amplitude):
    state = np.uint32(0x9E3779B9)
    noisy = []
    for value in signal:
        state = np.uint32(state * np.uint32(1_664_525) + np.uint32(1_013_904_223))
        unit = (float(state) / float(np.iinfo(np.uint32).max)) * 2.0 - 1.0
        noisy.append(float(value) + amplitude * unit)
    return np.asarray(noisy, dtype=np.float32)


def rmse(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    return float(np.sqrt(np.mean((left - right) ** 2)))


clean = synthetic_signal(n)
noisy = add_deterministic_noise(clean, noise_amplitude)
x = np.arange(n)

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(x, clean, color="black", lw=1.8, label="Clean")
ax.plot(x, noisy, color="tab:red", lw=0.9, alpha=0.75, label="Noisy")
ax.set(title="Clean signal and deterministic noisy observation", xlabel="Sample", ylabel="Amplitude")
ax.legend(loc="upper left")
plt.show()

print(f"Noisy RMSE: {rmse(clean, noisy):.4f}")

## Select the basis from coarsest-detail PDF shape

`select_basis` scores the coarsest retained detail packet. A narrower, more peaked distribution receives a higher score.

In [ ]:
selected = fcwt2.select_basis(noisy.tolist(), levels, candidates, "wavelet_packet")
scores = selected.candidate_scores()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)

axes[0].bar([score.wavelet for score in scores], [score.score for score in scores], color="tab:green", alpha=0.8)
axes[0].set(title="Basis score", ylabel="Score")
axes[0].tick_params(axis="x", rotation=30)

axes[1].scatter(
    [score.pdf_entropy for score in scores],
    [score.pdf_peak for score in scores],
    s=90,
    color="tab:purple",
)
for score in scores:
    axes[1].annotate(score.wavelet, (score.pdf_entropy, score.pdf_peak), xytext=(5, 4), textcoords="offset points")
axes[1].set(title="PDF shape diagnostics", xlabel="PDF entropy", ylabel="PDF peak mass")
plt.show()

print("Selected basis:", selected.selected)
for score in scores:
    print(
        f"{score.wavelet:>5} | score={score.score:8.3f} | "
        f"peak={score.pdf_peak:5.2f} | entropy={score.pdf_entropy:5.2f} | "
        f"MAD={score.median_abs_deviation:7.4f} | IQR={score.interquartile_range:7.4f}"
    )

## Decompose with the selected packet basis

The selected basis is fed back into `WaveletPacketTransform.with_wavelet(...)`.

In [ ]:
packet = fcwt2.WaveletPacketTransform.with_wavelet(levels, selected.selected)
tree = packet.decompose(noisy.tolist())
leaves = tree.leaves()
leaf_labels = ["".join(leaf.path) for leaf in leaves]
leaf_matrix = np.asarray([leaf.coefficients for leaf in leaves], dtype=float)

fig, ax = plt.subplots(figsize=(12, 5))
image = ax.imshow(np.abs(leaf_matrix), aspect="auto", origin="lower", interpolation="nearest", cmap="magma")
ax.set(title=f"Packet leaf magnitudes before thresholding ({packet.wavelet})", xlabel="Coefficient index", ylabel="Packet path")
ax.set_yticks(np.arange(len(leaf_labels)))
ax.set_yticklabels(leaf_labels, fontsize=8)
fig.colorbar(image, ax=ax, label="Magnitude")
plt.show()

## Estimate the threshold

This follows the Rust example: estimate noise scale from the median absolute detail coefficient, compute a universal threshold, then use a conservative multiplier.

In [ ]:
def has_detail(path):
    return any(band == "d" for band in path)


def universal_threshold(leaves, input_len):
    magnitudes = []
    for leaf in leaves:
        if has_detail(leaf.path):
            magnitudes.extend(np.abs(np.asarray(leaf.coefficients, dtype=float)))
    if not magnitudes:
        return 0.0
    median = float(np.median(magnitudes))
    sigma = median / 0.6745
    return sigma * np.sqrt(2.0 * np.log(input_len))


def soft_threshold(values, threshold):
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.maximum(np.abs(values) - threshold, 0.0)


threshold = 0.45 * universal_threshold(leaves, n)
print(f"Threshold: {threshold:.4f}")

## Modify leaves and reconstruct through `fcwt2`

Python code creates modified leaf data, but reconstruction still goes through `WaveletPacketTree.from_leaves(...)` and `tree.reconstruct()`.

In [ ]:
thresholded_leaf_data = []
thresholded_matrix = []

for leaf in leaves:
    coefficients = np.asarray(leaf.coefficients, dtype=float)
    if has_detail(leaf.path):
        coefficients = soft_threshold(coefficients, threshold)
    thresholded_leaf_data.append((list(leaf.path), coefficients.astype(np.float32).tolist()))
    thresholded_matrix.append(coefficients)

thresholded_tree = fcwt2.WaveletPacketTree.from_leaves(levels, thresholded_leaf_data, selected.selected)
denoised = np.asarray(thresholded_tree.reconstruct(), dtype=float)
thresholded_matrix = np.asarray(thresholded_matrix, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for ax, matrix, title in [
    (axes[0], np.abs(leaf_matrix), "Before thresholding"),
    (axes[1], np.abs(thresholded_matrix), "After thresholding"),
]:
    image = ax.imshow(matrix, aspect="auto", origin="lower", interpolation="nearest", cmap="magma")
    ax.set(title=title, xlabel="Coefficient index")
    ax.set_yticks(np.arange(len(leaf_labels)))
    ax.set_yticklabels(leaf_labels, fontsize=8)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
axes[0].set_ylabel("Packet path")
plt.show()

## Compare reconstruction quality

The deterministic setup makes the Rust and Python examples easy to compare. Small numeric differences can occur because the notebook uses NumPy operations and Python bindings, but the shape of the result should match.

In [ ]:
noisy_rmse = rmse(clean, noisy)
denoised_rmse = rmse(clean, denoised)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
axes[0].plot(x, clean, color="black", lw=1.6, label="Clean")
axes[0].plot(x, noisy, color="tab:red", lw=0.9, alpha=0.75, label=f"Noisy RMSE={noisy_rmse:.4f}")
axes[0].set(title="Noisy observation", ylabel="Amplitude")
axes[0].legend(loc="upper left")

axes[1].plot(x, clean, color="black", lw=1.6, label="Clean")
axes[1].plot(x, denoised, color="tab:blue", lw=1.1, label=f"Denoised RMSE={denoised_rmse:.4f}")
axes[1].set(title=f"Denoised with selected basis {selected.selected}", ylabel="Amplitude")
axes[1].legend(loc="upper left")

axes[2].plot(x, noisy - clean, color="tab:red", lw=0.9, alpha=0.7, label="Noisy error")
axes[2].plot(x, denoised - clean, color="tab:blue", lw=1.1, label="Denoised error")
axes[2].set(title="Error relative to clean signal", xlabel="Sample", ylabel="Error")
axes[2].legend(loc="upper left")
plt.show()

print("Selected basis:", selected.selected)
print(f"Threshold:     {threshold:.4f}")
print(f"Noisy RMSE:    {noisy_rmse:.4f}")
print(f"Denoised RMSE: {denoised_rmse:.4f}")

## What to try next

- Change `noise_amplitude` and rerun basis selection.
- Compare `wavelet_packet` scoring with `stationary` scoring.
- Try a hard-threshold variant to match the article experiment setup more closely.
- Inspect `score.pdf_peak`, `score.pdf_entropy`, `score.median_abs_deviation`, and `score.interquartile_range` for signals with different regularity.